In [3]:
# 1. Upload "Testing Bench Mark.zip", then enter both private credentials.
from pathlib import Path
import getpass, zipfile, shutil, subprocess, os, sys, importlib

ZIP_PATH = Path('/content/Testing Bench Mark.zip')
PROJECT_DIR = Path('/content/Testing_Bench_Mark')
YOLO_REPO = Path('/content/yolov5_v7')

assert ZIP_PATH.exists(), 'Upload Testing Bench Mark.zip to Colab first.'
archive_password = getpass.getpass('Archive password: ').encode('utf-8')
runtime_key = getpass.getpass('Runtime access key: ')

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    bad = z.testzip() if False else None
    z.extractall('/content', pwd=archive_password)

os.environ['TBM_ACCESS_KEY'] = runtime_key
os.environ['TBM_PROJECT_ROOT'] = str(PROJECT_DIR)
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'

if not YOLO_REPO.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'v7.0',
        'https://github.com/ultralytics/yolov5.git', str(YOLO_REPO)
    ], check=True)

%cd /content/Testing_Bench_Mark
%pip install -q -e /content/Testing_Bench_Mark
%pip install -q -r /content/yolov5_v7/requirements.txt

for module_name in list(sys.modules):
    if module_name == 'tbm_core_x7' or module_name.startswith('tbm_core_x7.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
from tbm_gate_x9.guard import assert_authorized
assert_authorized()
print('Protected project installed and access verified:', PROJECT_DIR)


/content/RCE_Unified_Benchmark_v4
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rce-unified-benchmark (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 126.6 MB/s eta 0:00:00
Project installed: /content/RCE_Unified_Benchmark_v4
Imported package: /content/RCE_Unified_Benchmark_v4/src/rce_benchmark/__init__.py
YOLOv5 release: /content/yolov5_v7


In [4]:
# 2. Download COCO only when it is not already present.
from pathlib import Path
import subprocess, zipfile

coco_root = Path('/content/coco')
coco_root.mkdir(parents=True, exist_ok=True)

downloads = {
    'val2017.zip': 'https://images.cocodataset.org/zips/val2017.zip',
    'annotations_trainval2017.zip': 'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
}

def download_verified_zip(url: str, path: Path) -> None:
    if path.exists() and not zipfile.is_zipfile(path):
        path.unlink()
    if not path.exists():
        print('Downloading', path.name)
        subprocess.run([
            'curl','-k','-L','--fail','--retry','5','--retry-delay','5',
            '--connect-timeout','30','--continue-at','-','--output',str(path),url
        ], check=True)
    if not zipfile.is_zipfile(path):
        path.unlink(missing_ok=True)
        raise RuntimeError(f'{path.name} is not a valid ZIP. Re-run the cell.')

for filename, url in downloads.items():
    archive = coco_root / filename
    expected = coco_root / ('val2017' if filename == 'val2017.zip' else 'annotations')
    if expected.exists():
        print('Already extracted:', expected)
        continue
    download_verified_zip(url, archive)
    print('Extracting', filename)
    with zipfile.ZipFile(archive) as z:
        z.extractall(coco_root)

print('COCO ready:', (coco_root/'val2017').exists(), (coco_root/'annotations/instances_val2017.json').exists())


Extracting val2017.zip
Extracting annotations_trainval2017.zip
COCO ready: True True


In [5]:
# 3. Generate or verify the candidate-pair manifest.
from pathlib import Path
import pandas as pd

manifest = Path('/content/coco_pairs.csv')
if not manifest.exists():
    !tbm-prepare \
      --annotations /content/coco/annotations/instances_val2017.json \
      --images /content/coco/val2017 \
      --output /content/coco_pairs.csv

frame = pd.read_csv(manifest)
print(f'Manifest rows: {len(frame):,}')
display(frame.head(3))


loading annotations into memory...
Done (t=0.78s)
creating index...
index created!
person+bicycle: 1520 candidate pairs
person+laptop: 450 candidate pairs
car+traffic light: 2150 candidate pairs
Wrote 4120 rows to /content/coco_pairs.csv
Manifest rows: 4,120


,image_id,image_path,object_a_name,object_a_coco_category_id,object_a_x1,object_a_y1,object_a_x2,object_a_y2,object_b_name,object_b_coco_category_id,object_b_x1,object_b_y1,object_b_x2,object_b_y2
0,5001,/content/coco/val2017/000000005001.jpg,person,1,10.790000,32.630001,69.029999,201.979996,bicycle,2,500.459991,159.100006,545.869995,258.359985
1,5001,/content/coco/val2017/000000005001.jpg,person,1,225.440002,255.639999,352.720001,480.000000,bicycle,2,500.459991,159.100006,545.869995,258.359985
2,5001,/content/coco/val2017/000000005001.jpg,person,1,77.250000,244.559998,190.550003,479.820007,bicycle,2,500.459991,159.100006,545.869995,258.359985


In [15]:
# 4. Create optimized per-detector configurations for a QUICK SMOKE TEST.
from pathlib import Path
import yaml, shutil

base = yaml.safe_load(Path('/content/Testing_Bench_Mark/02_CONFIGURATION/active/benchmark.yaml').read_text())
base['general']['manifest'] = '/content/coco_pairs.csv'

# --- TABDEELI YAHAN HAI ---
# Chhotay test ke liye rows ko sirf 5 par set kar diya hai
base['general']['maximum_rows'] = 5
base['general']['morf_steps'] = 10
base['general']['save_artifacts'] = True

# --- RCE Paper Specific Parameters (Table V & XI) ---
if 'RCE' not in base['methods']:
    base['methods']['RCE'] = {}
base['methods']['RCE']['num_concepts'] = 8         # K = 8
base['methods']['RCE']['temperature'] = 0.05       # tau_s = 0.05
base['methods']['RCE']['mask_percentile'] = 85     # Top 15% / 85th percentile
base['methods']['RCE']['blur_radius'] = 15         # sigma = 15

# Baqi baselines ki configurations
if 'D-RISE' in base['methods']:
    base['methods']['D-RISE']['num_masks'] = 1000
if 'GCE' in base['methods']:
    base['methods']['GCE']['max_training_images'] = 50
    base['methods']['GCE']['num_concepts'] = 15
    base['methods']['GCE']['vectors_per_image'] = 64

base['detectors']['yolov5x6']['repo'] = '/content/yolov5_v7'
base['detectors']['detr']['feature_layer'] = 'model.backbone.model.layer4'

run_root = Path('/content/rce_detector_runs')
if run_root.exists():
    shutil.rmtree(run_root)
run_root.mkdir(parents=True)

config_paths = {}

TARGET_DETECTOR = 'yolov5x6'

cfg = yaml.safe_load(yaml.safe_dump(base))
for name in cfg['detectors']:
    cfg['detectors'][name]['enabled'] = (name == TARGET_DETECTOR)
cfg['general']['output_directory'] = str(run_root / TARGET_DETECTOR)
path = Path(f'/content/benchmark_{TARGET_DETECTOR}.yaml')
path.write_text(yaml.safe_dump(cfg, sort_keys=False))

config_paths[TARGET_DETECTOR] = path
print(TARGET_DETECTOR, '->', path)


yolov5x6 -> /content/benchmark_yolov5x6.yaml


In [13]:
# 5. Detector-loader preflight.
import os, traceback, inspect
import tbm_core_x7
from tbm_core_x7.detectors.factory import build_detector

print('Using package:', tbm_core_x7.__file__)
assert 'Testing_Bench_Mark' in tbm_core_x7.__file__, 'Old v2/v3 package is loaded.'
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

preflight_configs = {
    'fasterrcnn': base['detectors']['fasterrcnn'],
    'yolov5x6': base['detectors']['yolov5x6'],
    'detr': base['detectors']['detr'],
}
preflight_status = {}
for name, cfg in preflight_configs.items():
    print(f'\nLoading {name} ...')
    try:
        model = build_detector(name, cfg, 'cuda' if __import__('torch').cuda.is_available() else 'cpu')
        extra = ''
        if name == 'detr':
            extra = f' | feature layer: {model.feature_layer_path}'
        print('OK:', name, extra)
        preflight_status[name] = 'OK'
        del model
        __import__('torch').cuda.empty_cache()
    except Exception as exc:
        preflight_status[name] = repr(exc)
        traceback.print_exc()

print('\nPreflight status:', preflight_status)
assert all(value == 'OK' for value in preflight_status.values()), 'A detector failed; check traceback.'


Using package: /content/RCE_Unified_Benchmark_v4/src/rce_benchmark/__init__.py

Loading fasterrcnn ...


YOLOv5 🚀 v7.0-0-g915bbf2 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)



OK: fasterrcnn 

Loading yolov5x6 ...


Fusing layers... 
YOLOv5x6 summary: 574 layers, 140730220 parameters, 0 gradients, 209.6 GFLOPs
Adding AutoShape... 


OK: yolov5x6 

Loading detr ...


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OK: detr  | feature layer: model.backbone.model.layer4

Preflight status: {'fasterrcnn': 'OK', 'yolov5x6': 'OK', 'detr': 'OK'}


In [16]:
# 6. Run each detector separately. One failed detector will not erase the others.
from pathlib import Path
import subprocess
import os

# PyTorch ki FutureWarnings ko chup (suppress) karwane ke liye
os.environ['PYTHONWARNINGS'] = 'ignore::FutureWarning'

run_status = {}
for detector_name, config_path in config_paths.items():
    print('\n' + '='*80)
    print('RUNNING:', detector_name)
    print('='*80)
    log_path = Path(f'/content/{detector_name}_benchmark.log')
    with log_path.open('w', encoding='utf-8') as log:
        # env=os.environ ka istemal taa-k warning filter subprocess par bhi apply ho
        process = subprocess.Popen(
            ['tbm-run', '--config', str(config_path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ
        )
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        code_value = process.wait()
    run_status[detector_name] = {'exit_code': code_value, 'log': str(log_path)}
    print(f'Exit code: {code_value}; log: {log_path}')

print('\nRun status:', run_status)



RUNNING: yolov5x6

=== Loading detector: yolov5x6 on cuda ===
YOLOv5 🚀 v7.0-0-g915bbf2 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5x6 summary: 574 layers, 140730220 parameters, 0 gradients, 209.6 GFLOPs
Adding AutoShape... 
Fitting GCE NMF dictionary for yolov5x6...
[yolov5x6] 1/5 5001_0 | D-RISE
[yolov5x6] 1/5 5001_0 | Finer-CAM
[yolov5x6] 1/5 5001_0 | GCE
[yolov5x6] 1/5 5001_0 | ODAM
[yolov5x6] 1/5 5001_0 | RCE
[yolov5x6] 2/5 5001_1 | D-RISE
[yolov5x6] 2/5 5001_1 | Finer-CAM
[yolov5x6] 2/5 5001_1 | GCE
[yolov5x6] 2/5 5001_1 | ODAM
[yolov5x6] 2/5 5001_1 | RCE
[yolov5x6] 3/5 5001_2 | D-RISE
[yolov5x6] 3/5 5001_2 | Finer-CAM
[yolov5x6] 3/5 5001_2 | GCE
[yolov5x6] 3/5 5001_2 | ODAM
[yolov5x6] 3/5 5001_2 | RCE
[yolov5x6] 4/5 5001_3 | D-RISE
[yolov5x6] 4/5 5001_3 | Finer-CAM
[yolov5x6] 4/5 5001_3 | GCE
[yolov5x6] 4/5 5001_3 | ODAM
[yolov5x6] 4/5 5001_3 | RCE
[yolov5x6] 5/5 5001_4 | D-RISE
[yolov5x6] 5/5 5001_4 | Finer-CAM
[yolov5x6] 5/5 5001_4 | 

In [17]:
# 7. Merge all available outputs and show errors.
from pathlib import Path
import pandas as pd
from tbm_core_x7.evaluation.statistics import aggregate_results
from tbm_core_x7.run import publication_table

run_root = Path('/content/rce_detector_runs')
frames = []
for detector_name in ['fasterrcnn', 'yolov5x6', 'detr']:
    path = run_root / detector_name / 'per_sample_results.csv'
    if path.exists():
        current = pd.read_csv(path)
        frames.append(current)
        print(detector_name, ':', len(current), 'rows')
    else:
        print(detector_name, ': per_sample_results.csv missing')

assert frames, 'No detector produced a diagnostic CSV.'
per_sample = pd.concat(frames, ignore_index=True)
final_root = Path('/content/rce_unified_results')
final_root.mkdir(parents=True, exist_ok=True)
per_sample.to_csv(final_root/'per_sample_results.csv', index=False)
aggregate = aggregate_results(per_sample)
aggregate.to_csv(final_root/'aggregate_results.csv', index=False)
table = publication_table(aggregate, ['yolov5x6','fasterrcnn','detr'])
table.to_csv(final_root/'table_vi_aoc.csv', index=False)

print('\nTABLE VI OUTPUT')
display(table)
errors = per_sample[per_sample['error'].notna()].copy()
print(f'Valid rows: {len(per_sample)-len(errors)} / {len(per_sample)}')
print(f'Error rows: {len(errors)}')
if not errors.empty:
    display(errors[['detector','method','image_id','error']].head(100))

print('Final files:', [p.name for p in final_root.iterdir()])


fasterrcnn : per_sample_results.csv missing
yolov5x6 : 25 rows
detr : per_sample_results.csv missing

TABLE VI OUTPUT


detector,Method,yolov5x6,fasterrcnn,detr,Average
0,D-RISE-adapted,0.050000,NaN,NaN,NaN
1,Finer-CAM-adapted,0.188692,NaN,NaN,NaN
2,GCE-adapted,0.553246,NaN,NaN,NaN
3,ODAM-adapted,0.170798,NaN,NaN,NaN
4,RCE,0.050277,NaN,NaN,NaN


Valid rows: 25 / 25
Error rows: 0
Final files: ['table_vi_aoc.csv', 'aggregate_results.csv', 'per_sample_results.csv']


In [ ]:
# 8. Download the result CSV files.
from google.colab import files
from pathlib import Path

root = Path('/content/rce_unified_results')
for name in ['table_vi_aoc.csv','aggregate_results.csv','per_sample_results.csv']:
    path = root / name
    if path.exists():
        files.download(str(path))
    else:
        print('Missing:', path)
